# Project 5: Rating the players of a team game

This notebook is the starting point of the project; the subject (`projects.pdf`, section 5)
gives the context and the grading. You hand in this notebook, completed: it must run from
top to bottom in less than 15 minutes on a laptop.

The project has three guided tasks, then a part where you choose how to go further:

1. **Elo for teams**: rate the players from the outcomes of $5$ against $5$ matches, and
   choose the step $k$.
2. **Newcomers**: a larger step for the players who just arrived.
3. **Matchmaking**: form balanced teams with the ratings, and measure what it changes.
4. **Going further**: a direction of your choice (see the end of the notebook).

Each task ends with a figure or a table and a short written answer (a few sentences) in the
cell "Answer". Everything outside the cells marked `...  # TODO` is given and working.

This notebook needs the version of `seqlearn` that contains `projects/`: if the imports below
fail, download the setup archive again from the course page and replace the folder
`seqlearn/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from seqlearn.plotting import COLORS, set_style
from seqlearn.preferences import kendall_tau, sigmoid
from seqlearn.projects.league import CAPACITY, TEAM_SIZE, make_league

set_style()

## 0. The league

An online game opposes two teams of $5$ players, in a league of $200$ active players. Each
player $i$ has a hidden strength $s_i$, and team $A$ beats team $B$ with probability
$\sigma\big(\sum_{i \in A} s_i - \sum_{j \in B} s_j\big)$: the Bradley-Terry model of
session 6, where the score of a team is the sum of the scores of its players.

The league moves: the strengths drift slowly; after each match, a player leaves with
probability $0.002$ and is replaced by a newcomer, who starts weak and improves over their
first $50$ matches or so (a few newcomers are strong from the start: experienced players
with a new account). Some players queue in parties of $2$ or $3$ friends, who must play in
the same team.

At each round, `league.queue()` returns $30$ waiting players, `league.party(i)` the party of
player $i$, and `league.play(team_a, team_b)` returns $1$ if team $A$ wins. The attributes
`strength`, `active` and `age` (number of matches played), and `win_probability`, are for the
evaluation only.

In [ ]:
league = make_league(rng=0)
queue = league.queue()
print("waiting players:", queue)
print("parties among them:", {tuple(league.party(i)) for i in queue if len(league.party(i)) > 1})

## Given: random teams, the season and the evaluation

A rating system is an object with two methods: `predict(team_a, team_b)` returns the
probability that $A$ wins, and `update(team_a, team_b, y)` learns from the outcome $y$. A
matchmaker is a function `matchmaker(league, queue, rng, system)` that returns two teams.

`season` plays $20\,000$ matches and measures:

- the **log-loss** of the predictions, computed *before* each match on the second half of
  the season: $-\log p$ if $A$ wins, $-\log(1 - p)$ otherwise (a constant prediction $1/2$
  gives $\log 2 \approx 0.69$; the true probabilities give about $0.47$ with random teams);
- the **Kendall correlation** between the ratings and the true strengths of the active
  players, averaged over the second half of the season;
- the **error on the newcomers** (between $10$ and $60$ matches played): the mean of
  $|r_i - s_i|$, after centering the ratings and the strengths on the active players;
- the **unfairness** of the matches, the mean of $|\mathbb{P}(A \text{ wins}) - 1/2|$.

In [ ]:
def random_teams(league, queue, rng, system=None):
    """Two teams of 5 drawn at random among the waiting players, parties kept together."""
    groups, seen = [], set()
    for i in queue:
        if i not in seen:
            seen.update(league.party(i))
            groups.append(league.party(i))
    order = rng.permutation(len(groups))
    a, b = [], []
    for k in order:
        g = groups[k]
        if len(a) + len(g) <= TEAM_SIZE:
            a += g
        elif len(b) + len(g) <= TEAM_SIZE:
            b += g
        if len(a) == len(b) == TEAM_SIZE:
            return a, b
    return None  # no split found: skip this round


def season(make_system, matchmaker=random_teams, seed=0, n_matches=20_000):
    league, rng, system = make_league(rng=seed), np.random.default_rng(seed + 100), make_system()
    losses, unfairness, tau_all, err_new = [], [], [], []
    for m in range(n_matches):
        teams = matchmaker(league, league.queue(), rng, system)
        if teams is None:
            continue
        a, b = teams
        p = system.predict(a, b)
        unfairness.append(abs(league.win_probability(a, b) - 0.5))
        y = league.play(a, b)
        system.update(a, b, y)
        if m >= n_matches // 2:
            losses.append(-np.log(p if y else 1 - p))
            if m % 500 == 0:
                active = league.active
                r, s = system.ratings[active], league.strength[active]
                tau_all.append(kendall_tau(s, r))
                new = (league.age[active] >= 10) & (league.age[active] <= 60)
                err_new.append(np.mean(np.abs((r - r.mean()) - (s - s.mean()))[new]))
    return {"log-loss": np.mean(losses), "Kendall": np.mean(tau_all),
            "error, newcomers": np.mean(err_new), "unfairness": np.mean(unfairness)}


def evaluate(systems, matchmaker=random_teams, n_seeds=3):
    """Table of the metrics (mean over the seeds) of each rating system."""
    table = {}
    for name, make_system in systems.items():
        runs = [season(make_system, matchmaker, seed) for seed in range(n_seeds)]
        table[name] = {key: np.mean([r[key] for r in runs]) for key in runs[0]}
        print(f"{name:<28}" + "  ".join(f"{key} {value:.3f}" for key, value in table[name].items()))
    return table

## Task 1: Elo for teams

Complete `TeamElo`: the rating of a team is the sum of the ratings $r_i$ of its players,
the prediction is $p = \sigma(R_A - R_B)$, and after the match each player of $A$ receives
$+k(y - p)$ and each player of $B$ receives $-k(y - p)$. Store the ratings in the array
`self.ratings` (indexed by the player ids, which go up to `CAPACITY`).

Then evaluate it with random teams for $k \in \{0.01, 0.03, 0.1, 0.3\}$ and plot the log-loss
and the Kendall correlation (all players) against $k$.

**Expected**: the figure, and an answer to: which $k$ is best, and why are a too small and a
too large step both bad in a league that moves? Compare the log-loss with the two references
($0.69$ and $0.47$).

**To read**: slides of session 6 (Elo, choosing the step); Elo, *The rating of chessplayers,
past and present*, 1978 (chapter 1, or the Wikipedia page "Elo rating system").

In [ ]:
class TeamElo:
    def __init__(self, k=0.03):
        self.k = k
        self.ratings = np.zeros(CAPACITY)

    def predict(self, team_a, team_b):
        ...  # TODO

    def update(self, team_a, team_b, y):
        ...  # TODO

In [ ]:
steps = [0.01, 0.03, 0.1, 0.3]
...  # TODO
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(steps, [row["log-loss"] for row in table1.values()], "o-", color=COLORS[0])
axes[0].axhline(np.log(2), color=COLORS[1], linestyle="--", linewidth=1, label="constant 1/2")
axes[0].axhline(0.47, color=COLORS[2], linestyle="--", linewidth=1, label="true probabilities")
axes[0].set(xscale="log", xlabel="step k", ylabel="log-loss", title="Task 1: predictions")
axes[0].legend()
axes[1].plot(steps, [row["Kendall"] for row in table1.values()], "o-", color=COLORS[0])
axes[1].set(xscale="log", xlabel="step k", ylabel="Kendall correlation", title="Task 1: ranking")
plt.show()

**Answer.** *Write your answer here.*

## Task 2: newcomers

A newcomer starts at rating $0$, but is weak at first and then improves. Complete
`NewcomerElo`: the same as `TeamElo`, but the players who have played fewer than $30$
matches in the system receive a step `k_new` instead of `k` (count the matches of each
player in an array). Evaluate `TeamElo(0.03)` and `NewcomerElo(0.03, k_new)` for
$k_\text{new} \in \{0.1, 0.3\}$.

**Expected**: the table printed by `evaluate`, and an answer to: on which players does it
help, and why? What would be a more principled way to choose the step of each player?
(Hint: keep an uncertainty on each rating, as Glicko and TrueSkill do.)

**To read**: Glickman, *Parameter estimation in large dynamic paired comparison
experiments*, JRSS C 1999 (the Glicko system, sections 1 and 2); Herbrich, Minka and
Graepel, *TrueSkill: a Bayesian skill rating system*, NeurIPS 2006 (sections 1 and 2).

In [ ]:
class NewcomerElo(TeamElo):
    def __init__(self, k=0.03, k_new=0.1, n_new=30):
        super().__init__(k)
        self.k_new, self.n_new = k_new, n_new
        self.matches = np.zeros(CAPACITY)

    def update(self, team_a, team_b, y):
        ...  # TODO

In [ ]:
...  # TODO

**Answer.** *Write your answer here.*

## Task 3: matchmaking

The matchmaker now uses the ratings. Complete `balanced_teams`: draw $30$ random splits with
`random_teams`, and return the one whose predicted probability `system.predict(a, b)` is the
closest to $1/2$. Evaluate `TeamElo(0.03)` with random and with balanced teams.

**Expected**: the table, and an answer to: how much fairer are the matches? Does the ranking
get better or worse, and why? Why can the log-loss not compare the two matchmakers? (Hint:
what does a good rating system predict for a balanced match?)

**To read**: the TrueSkill paper above (section on matchmaking); slides of session 6 (choosing
the duels: a duel between close models is the most informative).

In [ ]:
def balanced_teams(league, queue, rng, system, n_tries=30):
    ...  # TODO

In [ ]:
...  # TODO

**Answer.** *Write your answer here.*

## Going further

Choose one direction, among these suggestions or your own idea: state the question you ask,
then answer it as in the tasks (code, a figure, a short answer). One well-studied direction
is better than several superficial ones.

- A Gaussian rating per player (mean and variance), updated after each match as in Glicko or
  TrueSkill, with a step proportional to the variance.
- Bradley-Terry by maximum likelihood on a sliding window of matches: a logistic regression
  with features $\sum_{i \in A} e_i - \sum_{j \in B} e_j$ (session 6).
- The parties: can the ratings of two friends who always play together be told apart?
- The trade-off between fairness and information: balance the teams, but favour the
  matches with uncertain players.